In [1]:
from hwpx import HwpxDocument
from hwpx.tools.exporter import export_markdown_structured

with HwpxDocument.open("/home/maxjo/Work/python-hwpx/tests_new/input/07_청소년_대중문화예술인_또는_연습생_표준_부속합의서.hwpx") as doc:
    ...

manifest에서 masterPage를 찾지 못해 파일명 탐색 fallback을 사용합니다.
manifest에서 history를 찾지 못해 파일명 탐색 fallback을 사용합니다.
manifest에서 version 파트를 찾지 못해 기본 경로 fallback을 사용합니다: version.xml


In [2]:
export_markdown_structured(
        doc,
        # paragraph_level_only=True
    )

{'s1.p1.r2': '[별표1]',
 's1.p3.r1': '청소년 대중문화예술인(또는 연습생) 표준 부속합의서',
 's1.p4.r1.tbl1.tr2.tc1.p1.r1': '문화체육관광부고시 ',
 's1.p4.r1.tbl1.tr2.tc1.p2.r1': '제2025-0070호',
 's1.p4.r1.tbl1.tr3.tc1.p1.r1': '(2026. 01. 01. 개정)',
 's1.p6.r1.tbl1.tr1.tc1.p1.r1': '[대중문화예술기획업자]',
 's1.p6.r1.tbl1.tr1.tc3.p1.r1': '[와, 과]',
 's1.p6.r1.tbl1.tr2.tc1.p1.r1': '[대중문화예술인(또는 연습생)]',
 's1.p6.r1.tbl1.tr2.tc2.p1.r1': '(본명 :           )',
 's1.p6.r1.tbl1.tr2.tc3.p1.r1': '[는, 은]',
 's1.p6.r1.tbl1.tr3.tc1.p1.r1': '다음과 같이 합의서를 체결함에 있어 상호 신의성실로써 이를 이행한다.',
 's1.p8.r1': '제1조 (목적) ',
 's1.p8.r2': '이 부속합의서의 목적은 ',
 's1.p8.r3': '대중문화예술기획업자(이하 ‘기획업자’라 한다)와 청소년 대중문화예술인(또는 청소년 연습생, 이하 ‘대중문화예술인’이라 한다)',
 's1.p8.r4': ' 사이에 ',
 's1.p8.r5': '________________________________ ',
 's1.p8.r6': '(이하 ‘주계약’ 이라 한다)을 체결함에 있어서 청소년의 ',
 's1.p8.r7': '권익을 보다 명확하게 보호',
 's1.p8.r8': '하고 청소년이 ',
 's1.p8.r9': '건전한 인격체로 성장할 수 있도록 지원하는데 ',
 's1.p8.r10': '필요한 사항을 정하는 것에',
 's1.p8.r11': ' 있다.',
 's1.p8.r12': ' ',
 's1.p9.r1': '※ 위 빈칸에 기획업자와 대중문화예술인 사이에 

|Korean  |English Term|Example Citation|
|--------|------------|----------------|
|조 (Jo)  |Article     |Article 1       |
|항 (Hang)|Paragraph   |Paragraph (1)   |
|호 (Ho)  |Subparagraph|Subparagraph 1. |
|목 (Mok) |Item        |Item (a)        |


In [3]:
import re
from typing import TypedDict

class NumberMatch(TypedDict):
    val: str
    span: tuple[int, int]

def get_article_numbers(text: str) -> list[NumberMatch]:
    # Base matcher for "제N조". We then filter to likely article headings.
    pattern = r"제\s*(\d+)\s*조"
    results: list[NumberMatch] = []
    postposition_chars = set("에의을를은는이가와과로도만")

    for match in re.finditer(pattern, text):
        start, end = match.span()
        next_char = text[end:end + 1]

        # Ignore cited forms like "제8조에", "제8조의", or "제8조를".
        if next_char and next_char in postposition_chars:
            continue

        prev_text = text[:start].rstrip()
        next_text = text[end:].lstrip()

        preceded_like_heading = not prev_text or prev_text[-1] in ".!?)]}"
        followed_like_heading = not next_text or next_text.startswith(("(", "[", "<"))

        if preceded_like_heading or (start == 0) or followed_like_heading:
            results.append({
                "val": match.group(1),
                "span": match.span()  # (start, end)
            })
    return results

def _enclosed_number_value(ch: str) -> int | None:
    cp = ord(ch)
    if 0x2460 <= cp <= 0x2473:  # ①..⑳
        return cp - 0x2460 + 1
    if 0x2474 <= cp <= 0x2487:  # ⑴..⒇
        return cp - 0x2474 + 1
    if 0x2488 <= cp <= 0x249B:  # ⒈..⒛
        return cp - 0x2488 + 1
    if cp == 0x24EA:  # ⓪
        return 0
    if 0x24F5 <= cp <= 0x24FE:  # ⓵..⓾
        return cp - 0x24F5 + 1
    if 0x2776 <= cp <= 0x277F:  # ❶..❿
        return cp - 0x2776 + 1
    if 0x2780 <= cp <= 0x2789:  # ➀..➉
        return cp - 0x2780 + 1
    if 0x278A <= cp <= 0x2793:  # ➊..➓
        return cp - 0x278A + 1
    return None

def get_paragraph_numbers(text: str) -> list[NumberMatch]:
    # Group 1: (digits) | Group 2: enclosed Unicode number variants
    pattern = r"\((\d+)\)|([\u2460-\u2473\u2474-\u2487\u2488-\u249B\u24EA\u24F5-\u24FE\u2776-\u277F\u2780-\u2789\u278A-\u2793])"
    results: list[NumberMatch] = []

    for match in re.finditer(pattern, text):
        if match.group(1):
            val = match.group(1)
        else:
            number = _enclosed_number_value(match.group(2))
            if number is None:
                continue
            val = str(number)

        results.append({
            "val": val,
            "span": match.span(),
        })
    return results

그 dynamic 할당 안되어있음 최고단 for loop에 대해서, article nums에 대한 순환 필요!!!!

In [4]:
from pydantic import BaseModel, Field

class IR(BaseModel):
    raw_text: str = ""
    markdown_text: str = ""
    section: str = "unk"
    article_n: list[str] = Field(default_factory=list)
    paragraph_n: list[str] = Field(default_factory=list)
    splits: list[tuple[int, int]] = Field(default_factory=list)  # "조" (article) 기준 쪼사기


def create_ir_dict(doc: HwpxDocument) -> dict[str, IR]:
    parsed = export_markdown_structured(doc)
    irs: dict[str, IR] = {}
    active_article_num: str = "-1"
    active_paragraph_num: str | None = None

    for id, text in parsed.items():
        detected_article_nums: list[str] = []
        detected_paragraph_nums: list[str] = []
        splits: list[tuple[int, int]] = []
        section = "unk-table" if "tbl" in id else "unk"

        ex_article = get_article_numbers(text)
        for match in ex_article:
            detected_article_nums.append(match["val"])
            splits.append(match["span"])

        # Paragraph markers before the first article belong to the previous article.
        prefix_end = splits[0][0] if splits else len(text)
        for match in get_paragraph_numbers(text[:prefix_end]):
            if active_article_num != "-1":
                detected_paragraph_nums.append(f"{active_article_num}.{match['val']}")

        # Paragraph markers after each article heading belong to that article.
        for i, article_num in enumerate(detected_article_nums):
            seg_start = splits[i][1]
            seg_end = splits[i + 1][0] if i + 1 < len(splits) else len(text)
            for match in get_paragraph_numbers(text[seg_start:seg_end]):
                detected_paragraph_nums.append(f"{article_num}.{match['val']}")

        article_nums = detected_article_nums.copy()
        paragraph_nums = detected_paragraph_nums.copy()

        if detected_paragraph_nums and not detected_article_nums and active_article_num != "-1":
            article_nums = [active_article_num]

        if not detected_article_nums and not detected_paragraph_nums:
            if active_article_num != "-1":
                article_nums = [active_article_num]
            if active_paragraph_num is not None:
                paragraph_nums = [active_paragraph_num]

        irs[id] = IR(
            raw_text=text,
            markdown_text=text,
            section=section,
            article_n=article_nums,
            paragraph_n=paragraph_nums,
            splits=splits,
        )

        if detected_paragraph_nums:
            active_paragraph_num = detected_paragraph_nums[-1]
            active_article_num = active_paragraph_num.split('.', 1)[0]
        elif detected_article_nums:
            active_article_num = detected_article_nums[-1]
            active_paragraph_num = None

    return irs

In [5]:
from pprint import pprint

irs = create_ir_dict(doc)
[pprint({k: ir.model_dump()}) for k, ir in irs.items()]
print()

{'s1.p1.r2': {'article_n': [],
              'markdown_text': '[별표1]',
              'paragraph_n': [],
              'raw_text': '[별표1]',
              'section': 'unk',
              'splits': []}}
{'s1.p3.r1': {'article_n': [],
              'markdown_text': '청소년 대중문화예술인(또는 연습생) 표준 부속합의서',
              'paragraph_n': [],
              'raw_text': '청소년 대중문화예술인(또는 연습생) 표준 부속합의서',
              'section': 'unk',
              'splits': []}}
{'s1.p4.r1.tbl1.tr2.tc1.p1.r1': {'article_n': [],
                                 'markdown_text': '문화체육관광부고시 ',
                                 'paragraph_n': [],
                                 'raw_text': '문화체육관광부고시 ',
                                 'section': 'unk-table',
                                 'splits': []}}
{'s1.p4.r1.tbl1.tr2.tc1.p2.r1': {'article_n': [],
                                 'markdown_text': '제2025-0070호',
                                 'paragraph_n': [],
                                 'raw_text': '제2025-0070호

In [6]:
from collections import defaultdict
from tabulate import tabulate


def _sorted_ir_items(irs: dict[str, IR]) -> list[tuple[str, IR]]:
    def sort_key(item: tuple[str, IR]) -> tuple[int, ...]:
        id, _ = item
        return tuple(int(num) for num in re.findall(r"\d+", id))

    return sorted(irs.items(), key=sort_key)


def table_formatter(irs: dict[str, IR]) -> str:
    table_ids = {
        match.group(1)
        for id in irs
        if (match := re.match(r"^(.*?\.tbl\d+)", id))
    }

    assert table_ids, "table_formatter expects table IR entries"
    assert len(table_ids) == 1, "table_formatter expects entries from exactly one table"

    cell_runs: dict[tuple[int, int, int], list[tuple[int, str]]] = defaultdict(list)
    for id, ir in _sorted_ir_items(irs):
        match = re.search(r"\.tr(\d+)\.tc(\d+)\.p(\d+)(?:\.r(\d+))?$", id)
        if not match:
            continue

        row_num = int(match.group(1))
        col_num = int(match.group(2))
        para_num = int(match.group(3))
        run_num = int(match.group(4) or 1)
        cell_runs[(row_num, col_num, para_num)].append((run_num, ir.raw_text))

    rows: dict[int, dict[int, str]] = defaultdict(dict)
    for (row_num, col_num, para_num), runs in sorted(cell_runs.items()):
        text = "".join(run_text for run_num, run_text in sorted(runs)).strip()
        if para_num > 1 and rows[row_num].get(col_num):
            rows[row_num][col_num] += "<br>" + text
        else:
            rows[row_num][col_num] = text

    max_col = max(max(cols) for cols in rows.values())
    table = [
        [rows[row_num].get(col_num, "") for col_num in range(1, max_col + 1)]
        for row_num in sorted(rows)
    ]
    return tabulate(table, tablefmt="github")

In [7]:
print(table_formatter({id: ir for id, ir in irs.items() if "s1.p44.r1.tbl1" in id}))

|--------------------------|---------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------|
| 구 분                    | 대중문화예술용역 제공시간 제한                                                              | 대중문화예술용역 제공시간 제한                                                                                                                |
| 15세<br>미만의<br>청소년 | ㆍ 주당 35시간 이내                                                                         | ㆍ 오후 10시에서 오전 6시까지 금지<br>* 단, 용역 제공일의 다음 날이 학교의 휴일인 경우에는 청소년과 법정대리인의 동의 하에 자정까지 제공 가능 |
| 15세<br>이상의<br>청소년 | ㆍ 주당 40시간 이내<br>* 단, 청소년이 합의할 경우에는 1일 1시간, 1주일 6시간 한도 연장 가능 | ㆍ 오후 10시부터 오전 6시까지 금지<br>* 단, 청소년과 법정대리인이 동의하는 경우에는 오후 10시에서 오전 6시 사이에도 제공 가능                 |


In [8]:
class ArticleIR(BaseModel):
    formatted_str: str = ""
    article_n: str = "-1"
    IRchunk_ids: list[str] = Field(default_factory=list)
    IRchunks: list[IR] = Field(default_factory=list)  # original IRs used in-order
    IRjoin: list[int] = Field(default_factory=list)  # formatted_str index of where the IRs were joined
    IRtrim: tuple[int, int] = (0, 0)  # outer IRs(left, right) raw_text index where it was trimmed (if trimmed)

    @staticmethod
    def _normalize_enclosed_numbers(text: str) -> str:
        return re.sub(
            r"[\u2460-\u2473\u2474-\u2487\u2488-\u249B\u24EA\u24F5-\u24FE\u2776-\u277F\u2780-\u2789\u278A-\u2793]",
            lambda m: f"({_enclosed_number_value(m.group(0))})",
            text,
        )

    def export_to_markdown(self) -> str:
        # Rebuild from chunks to keep table/text formatting deterministic.
        if not self.IRchunks or not self.IRchunk_ids:
            return self._normalize_enclosed_numbers(self.formatted_str)

        parts: list[str] = []
        table_roots_emitted: set[str] = set()
        join_points: list[int] = []

        for id, ir in zip(self.IRchunk_ids, self.IRchunks):
            if ".tbl" in id:
                table_root = re.match(r"^(.*?\.tbl\d+)", id)
                if not table_root:
                    continue

                root = table_root.group(1)
                if root in table_roots_emitted:
                    continue

                table_subset = {
                    chunk_id: chunk_ir
                    for chunk_id, chunk_ir in zip(self.IRchunk_ids, self.IRchunks)
                    if chunk_id.startswith(root)
                }
                if table_subset:
                    join_points.append(sum(len(part) for part in parts))
                    parts.append("\n" + table_formatter(table_subset) + "\n")
                    table_roots_emitted.add(root)
            else:
                join_points.append(sum(len(part) for part in parts))
                parts.append(ir.raw_text)

        self.IRjoin = join_points
        self.formatted_str = "".join(parts).strip()
        return self._normalize_enclosed_numbers(self.formatted_str)


def article_former(irs: dict[str, IR]) -> list[ArticleIR]:
    articles: list[ArticleIR] = []
    current: ArticleIR | None = None

    for id, ir in _sorted_ir_items(irs):
        article_n = ir.article_n[0] if ir.article_n else "-1"
        if current is None or current.article_n != article_n:
            current = ArticleIR(article_n=article_n)
            articles.append(current)

        current.IRchunk_ids.append(id)
        current.IRchunks.append(ir)

    for article in articles:
        article.formatted_str = article.export_to_markdown()

    return articles


In [9]:
[print(a.formatted_str) for a in article_former(irs)]
print()

[별표1]청소년 대중문화예술인(또는 연습생) 표준 부속합의서
|-------------------------------------|
| 문화체육관광부고시<br>제2025-0070호 |
| (2026. 01. 01. 개정)                |

|---------------------------------------------------------------------|---------------------|----------|
| [대중문화예술기획업자]                                              |                     | [와, 과] |
| [대중문화예술인(또는 연습생)]                                       | (본명 :           ) | [는, 은] |
| 다음과 같이 합의서를 체결함에 있어 상호 신의성실로써 이를 이행한다. |                     |          |
제1조 (목적) 이 부속합의서의 목적은 대중문화예술기획업자(이하 ‘기획업자’라 한다)와 청소년 대중문화예술인(또는 청소년 연습생, 이하 ‘대중문화예술인’이라 한다) 사이에 ________________________________ (이하 ‘주계약’ 이라 한다)을 체결함에 있어서 청소년의 권익을 보다 명확하게 보호하고 청소년이 건전한 인격체로 성장할 수 있도록 지원하는데 필요한 사항을 정하는 것에 있다. ※ 위 빈칸에 기획업자와 대중문화예술인 사이에 체결한 주계약의 정확한 명칭을 기재하십시오.
제2조 (적용) 이 부속 합의서는 별도의 계약을 구성하고 있으며, 주계약 보다 우선 적용된다.
제3조 (청소년의 자유선택권 보장) 대중문화예술인은 기획업자에 대하여 자기 의사를 자유롭게 밝히고 스스로 결정할 권리를 가지며, 기획업자는 대중문화예술인의 자유 선택권을 침해하지 않아야 한다.
제4조 (청소년의 학습권 보장) (1) 기획업자는 대중문화예술인이 「교육기본법」 제8조에 따른 의무교

In [10]:
import os

inp_dir = "/home/maxjo/Work/python-hwpx/tests_new/input/"
inp_files = [inp_dir + fname for fname in os.listdir(inp_dir)]
inp_files

['/home/maxjo/Work/python-hwpx/tests_new/input/01_대중문화예술분야_연습생_표준계약서.hwpx',
 '/home/maxjo/Work/python-hwpx/tests_new/input/03_웹소설_분야_표준계약서_고시_제정안.hwpx',
 '/home/maxjo/Work/python-hwpx/tests_new/input/04_만화_분야_표준계약서_고시_제2024-0032호_2024-06-13_개정.hwpx',
 '/home/maxjo/Work/python-hwpx/tests_new/input/05_방송프로그램_제작_표준계약서_방송_분야.hwpx',
 '/home/maxjo/Work/python-hwpx/tests_new/input/06_국제회의용역_표준계약서_고시_문체부_제2022-45호.hwpx',
 '/home/maxjo/Work/python-hwpx/tests_new/input/07_청소년_대중문화예술인_또는_연습생_표준_부속합의서.hwpx',
 '/home/maxjo/Work/python-hwpx/tests_new/input/reference-testfile.hwpx']

In [11]:
import tiktoken

encoder = tiktoken.get_encoding("o200k_base")

with HwpxDocument.open(inp_files[5]) as doc:
    ir_dict = create_ir_dict(doc)
    fmt_str = [a.formatted_str for a in article_former(ir_dict)]
    [print(s) for s in fmt_str]
    print("\n", 25*"=", "\n")
    print([len(encoder.encode(s)) for s in fmt_str])

manifest에서 masterPage를 찾지 못해 파일명 탐색 fallback을 사용합니다.
manifest에서 history를 찾지 못해 파일명 탐색 fallback을 사용합니다.
manifest에서 version 파트를 찾지 못해 기본 경로 fallback을 사용합니다: version.xml


[별표1]청소년 대중문화예술인(또는 연습생) 표준 부속합의서
|-------------------------------------|
| 문화체육관광부고시<br>제2025-0070호 |
| (2026. 01. 01. 개정)                |

|---------------------------------------------------------------------|---------------------|----------|
| [대중문화예술기획업자]                                              |                     | [와, 과] |
| [대중문화예술인(또는 연습생)]                                       | (본명 :           ) | [는, 은] |
| 다음과 같이 합의서를 체결함에 있어 상호 신의성실로써 이를 이행한다. |                     |          |
제1조 (목적) 이 부속합의서의 목적은 대중문화예술기획업자(이하 ‘기획업자’라 한다)와 청소년 대중문화예술인(또는 청소년 연습생, 이하 ‘대중문화예술인’이라 한다) 사이에 ________________________________ (이하 ‘주계약’ 이라 한다)을 체결함에 있어서 청소년의 권익을 보다 명확하게 보호하고 청소년이 건전한 인격체로 성장할 수 있도록 지원하는데 필요한 사항을 정하는 것에 있다. ※ 위 빈칸에 기획업자와 대중문화예술인 사이에 체결한 주계약의 정확한 명칭을 기재하십시오.
제2조 (적용) 이 부속 합의서는 별도의 계약을 구성하고 있으며, 주계약 보다 우선 적용된다.
제3조 (청소년의 자유선택권 보장) 대중문화예술인은 기획업자에 대하여 자기 의사를 자유롭게 밝히고 스스로 결정할 권리를 가지며, 기획업자는 대중문화예술인의 자유 선택권을 침해하지 않아야 한다.
제4조 (청소년의 학습권 보장) (1) 기획업자는 대중문화예술인이 「교육기본법」 제8조에 따른 의무교

```bash
curl http://sylph-wsl:8000/v1/chat/completions \
-H "Content-Type: application/json" \
-d @curl_payload.json
```

In [16]:
import requests
from tqdm import tqdm

URL = "http://sylph-wsl:8000/v1/chat/completions"
MODEL_NAME = "jinkyeongk/Midm-2.0-Base-Instruct-AWQ"

BASE_PROMPT = """당신은 계약서 조문 분류기입니다. 반드시 아래 형식으로만 답하세요:

  GOOD
  또는
  NG: [이유 한 줄]

  [판단 기준]
  - GOOD: 조문(제N조) 내용만 있거나, 조문을 보조하는 표·주석이 포함된 경우
    · 조문 문장 안에 빈칸(___) 또는 등록번호·계약명 기재란이 있어도 GOOD
  - NG: 아래 중 하나라도 조문 외부에 독립적으로 존재하는 경우
    · 문서 제목, 고시 번호, 개정일
    · 계약체결 일시·장소 작성란 (독립 항목으로 존재)
    · 서명란 (주소, 생년월일, 성명, 인 등이 독립 블록으로 존재)
    · 당사자 표 (기획업자/대중문화예술인 서명 블록)

  다른 말은 절대 하지 마세요. 반드시 GOOD 또는 NG: 로 시작하세요.
"""

payload = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "system",
            "content": ""
        },
        {
			"role": "user",
            "content": ""
		}
    ]
}

def send_chat_request(userprompt, systemprompt=BASE_PROMPT):
    payload["messages"][0]["content"] = systemprompt
    payload["messages"][1]["content"] = userprompt
    try:
        response = requests.post(URL, json=payload)
        
        response.raise_for_status()
        
        result = response.json()
        return result['choices'][0]['message']['content']
        
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

for a in fmt_str:
    resp = send_chat_request(a)
    print("내용:", a)
    print()
    print("답변:", resp)
    print('\n\n\n', 25*"=", '\n')


내용: [별표1]청소년 대중문화예술인(또는 연습생) 표준 부속합의서
|-------------------------------------|
| 문화체육관광부고시<br>제2025-0070호 |
| (2026. 01. 01. 개정)                |

|---------------------------------------------------------------------|---------------------|----------|
| [대중문화예술기획업자]                                              |                     | [와, 과] |
| [대중문화예술인(또는 연습생)]                                       | (본명 :           ) | [는, 은] |
| 다음과 같이 합의서를 체결함에 있어 상호 신의성실로써 이를 이행한다. |                     |          |

답변: NG: 문서 제목, 고시 번호, 개정일이 조문 외부에 독립적으로 존재하여 조문 분류 기준에 맞지 않습니다.




내용: 제1조 (목적) 이 부속합의서의 목적은 대중문화예술기획업자(이하 ‘기획업자’라 한다)와 청소년 대중문화예술인(또는 청소년 연습생, 이하 ‘대중문화예술인’이라 한다) 사이에 ________________________________ (이하 ‘주계약’ 이라 한다)을 체결함에 있어서 청소년의 권익을 보다 명확하게 보호하고 청소년이 건전한 인격체로 성장할 수 있도록 지원하는데 필요한 사항을 정하는 것에 있다. ※ 위 빈칸에 기획업자와 대중문화예술인 사이에 체결한 주계약의 정확한 명칭을 기재하십시오.

답변: GOOD




내용: 제2조 (적용) 이 부속 합의서는 별도의 계약을 구성하고 있으며, 주계약 보다 우선 적용된다.

답변: GOOD




내용: 제3조 (청소년의 자유선택권 보장) 대중문화예술인은 기획업자에 대하여 자기 의사를 자유롭게 

In [13]:
send_chat_request("")

'NG: 문서 제목, 고시 번호, 개정일 등이 포함되어 있어 조문 분류기의 판단 기준에 맞지 않습니다.'